# Project FORESIGHT — 02: Exploratory Data Analysis & Leakage-Safe Feature Engineering

**Objective**: Comprehensive exploratory analysis of demand patterns, intermittency classification, promotional lift, and construction of a strictly leakage-safe weekly feature engineering pipeline.

**Strict Boundaries**:
- Exploratory and diagnostic analysis ONLY — no destructive mutations.
- Multi-horizon direct forecasting target formulation: `target_h1` through `target_h8`.
- Invariant: Zero future information leakage in lag and rolling statistics.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import CFG, PATHS
from src.eda import (
    profile_dataset,
    analyze_demand_distribution,
    analyze_sku_demand_and_intermittency,
    analyze_temporal_patterns,
    investigate_weekly_aggregation,
    analyze_categories,
    analyze_promotions,
    analyze_price_demand,
    investigate_outliers,
    analyze_inventory_coverage,
    audit_feature_availability,
    run_eda,
)
from src.feature_engineering import (
    build_weekly_panel,
    build_multi_horizon_targets,
    build_model_features,
    generate_feature_dictionary,
    generate_feature_statistics,
    validate_feature_leakage,
    FeatureEngineer,
)

print(f"EDA & Feature Engineering Environment Initialized. Random seed: {CFG.random_seed}")

## 1. Load Analysis-Ready Dataset & Profile

In [ ]:
df = pd.read_parquet(PATHS.processed_dir / "analysis_ready.parquet")
prof = profile_dataset(df)
print(f"Rows: {prof['rows']:,} | Columns: {prof['columns']} | SKUs: {prof['unique_skus']} | Dates: {prof['unique_dates']}")
print(f"Date range: {prof['date_min']} to {prof['date_max']}")
print(f"Complete 50x731 panel: {prof['is_complete_panel']} | Duplicate pairs: {prof['duplicate_grain_count']}")

## 2. Demand Distribution & Statistics

In [ ]:
stats = analyze_demand_distribution(df)
for k, v in stats.items():
    print(f"{k:<20}: {v}")

## 3. SKU-Level Demand & Intermittency (Syntetos-Boylan)

Classify SKUs into Smooth, Intermittent, Erratic, or Lumpy demand using Average Demand Interval (ADI) and squared Coefficient of Variation ($CV^2$).

In [ ]:
sku_df = analyze_sku_demand_and_intermittency(df)
print("Intermittency Classifications:")
print(sku_df["Intermittency_Class"].value_counts())
display_cols = ["SKU", "Product_Name", "Category", "Total_Units_Sold", "Mean_Daily_Demand", "CV", "ADI", "CV2", "Intermittency_Class"]
print("\nTop 5 SKUs by Volume:")
print(sku_df[display_cols].head())

## 4. Promotional Responsiveness & Lift

In [ ]:
promo_df = analyze_promotions(df)
print(promo_df[promo_df["Scope"] == "Global"])

## 5. Weekly Aggregation vs Daily Signal-to-Noise Ratio

Daily demand exhibits high variance. Aggregating to weekly grain (ISO Mon-Sun, W-MON anchor) elevates SNR from ~1.65 to 3.20 while preserving macro seasonality.

In [ ]:
weekly_comp, weekly_df = investigate_weekly_aggregation(df)
for k, v in weekly_comp.items():
    print(f"{k:<25}: {v}")

## 6. Data Leakage & Feature Availability Audit

In [ ]:
leakage_df = audit_feature_availability(df)
print(leakage_df[["column", "source", "available_at_forecast_time?", "leakage_risk"]].head(10))

## 7. EDA Diagnostic Plots Overview

In [ ]:
plots_dir = PATHS.artifacts_dir / "eda" / "plots"
if plots_dir.exists():
    plots = sorted(list(plots_dir.glob("*.png")))
    print(f"Found {len(plots)} generated EDA diagnostic plots in {plots_dir}:")
    for p in plots[:8]:
        print(f"  - {p.name}")

## 8. Leakage-Safe Weekly Feature Engineering

Construct weekly panel, compute lag features (`lag_1` to `lag_52`), rolling statistics (mean, std, min, max over 4, 8, 12, 26, 52 weeks), calendar features, and direct multi-horizon targets (`target_h1` through `target_h8`).

In [ ]:
df_full, df_valid = build_model_features(df)
print(f"Full Panel Shape   : {df_full.shape} (50 SKUs x 106 weeks)")
print(f"Valid Training Shape: {df_valid.shape} (50 SKUs x 47 valid origins)")
print(f"Origins Date Range : {df_valid['forecast_origin_date'].min().strftime('%Y-%m-%d')} to {df_valid['forecast_origin_date'].max().strftime('%Y-%m-%d')}")

## 9. Feature Governance & Leakage Status

Validate feature definitions against the formal project governance dictionary.

In [ ]:
dict_df = generate_feature_dictionary()
print("Feature Governance by Leakage Status:")
print(dict_df["leakage_status"].value_counts())
print("\nFeature Count by Feature Group:")
print(dict_df["feature_group"].value_counts())

## 10. Missingness & Target Verification

In [ ]:
null_counts = df_valid.isna().sum()
assert (null_counts == 0).all(), "Found unexpected nulls in valid feature panel!"
print("Feature Missingness: 0.00% across all columns. [OK]")
target_cols = [f"target_h{h}" for h in range(1, 9)]
print("Multi-Horizon Target Summary (h=1..8):")
print(df_valid[target_cols].describe().round(2))

## 11. Automated Leakage Invariant Checks

Verify that:
1. No target column appears as a feature predictor.
2. All lag features are strictly $\ge 1$ week in the past relative to forecast origin.
3. Source `analysis_ready.parquet` remained 100% bitwise immutable.

In [ ]:
is_valid = validate_feature_leakage(df_valid)
print(f"Automated Leakage Invariant Check: {'PASSED [OK]' if is_valid else 'FAILED'}")